# Object Detection with Phi-4-Multimodal

Images resized to **500×500**. Model asked to detect **at most 4 objects** with bounding boxes.
Includes retry logic and regex fallback for malformed JSON.

In [ ]:
import os, json, re, base64
import cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import UserMessage, TextContentItem, ImageContentItem, ImageUrl
from azure.core.credentials import AzureKeyCredential

load_dotenv()
client = ChatCompletionsClient(
    endpoint=os.environ['AZURE_ENDPOINT'],
    credential=AzureKeyCredential(os.environ['AZURE_API_KEY']),
)
IMG_SIZE = 500
COLORS = ['#00FF00','#FF0000','#0000FF','#FFFF00','#FF00FF','#00FFFF','#80FF00','#FF8000']

DETECTION_PROMPT = (
    f'This image is {IMG_SIZE}x{IMG_SIZE} pixels. Detect at most 4 objects. '
    f'For each object return label and bounding box as pixel coordinates.\n\n'
    f'Return ONLY a JSON array like this (values are for reference only, use actual values from the image but keep the structure the same):\n'
    f'[{{"label":"dog","bbox":[150,100,350,400]}},{{"label":"ball","bbox":[390,340,450,400]}}]'
)
print(f'Ready. Prompt asks for max 4 objects on {IMG_SIZE}x{IMG_SIZE} images.')

In [ ]:
def resize_and_encode(path):
    img = cv2.imread(path)
    resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode('.jpg', resized, [cv2.IMWRITE_JPEG_QUALITY, 90])
    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB), base64.b64encode(buf.tobytes()).decode('utf-8')

def normalize(dets):
    for d in dets:
        bbox = d['bbox']
        mx = max(bbox) if bbox else 0
        if mx <= 1.0: d['bbox'] = [v * IMG_SIZE for v in bbox]
        elif mx <= 100: d['bbox'] = [v / 100 * IMG_SIZE for v in bbox]
        d['bbox'] = [max(0, min(IMG_SIZE, v)) for v in d['bbox']]
    return dets

def parse(raw):
    # Clean JSON
    try:
        p = json.loads(raw)
        if isinstance(p, dict):
            for v in p.values():
                if isinstance(v, list): p = v; break
        if isinstance(p, list):
            d = [x for x in p if isinstance(x, dict) and 'label' in x and 'bbox' in x]
            if d: return normalize(d)
    except json.JSONDecodeError: pass
    # Extract [...]
    m = re.search(r'\[.*\]', raw, re.DOTALL)
    if m:
        try:
            d = [x for x in json.loads(m.group()) if isinstance(x, dict) and 'label' in x and 'bbox' in x]
            if d: return normalize(d)
        except json.JSONDecodeError: pass
    # Regex fallback
    pairs = re.findall(r'"label"\s*:\s*"([^"]+)".*?"bbox"\s*:\s*\[([\d\s,.]+)\]', raw, re.DOTALL)
    if pairs:
        seen, dets = set(), []
        for label, bs in pairs:
            if label in seen: continue
            nums = [float(x.strip()) for x in bs.split(',') if x.strip()]
            if len(nums) == 4: seen.add(label); dets.append({'label': label, 'bbox': nums})
        if dets: return normalize(dets)
    return []

def detect_objects(path, max_retries=3):
    rgb, b64 = resize_and_encode(path)
    for attempt in range(1, max_retries + 1):
        r = client.complete(
            messages=[UserMessage(content=[
                TextContentItem(text=DETECTION_PROMPT),
                ImageContentItem(image_url=ImageUrl(url=f'data:image/jpeg;base64,{b64}')),
            ])], max_tokens=512, temperature=0.2,
        )
        raw = r.choices[0].message.content.strip()
        print(f'[Attempt {attempt}] {raw[:150]}')
        dets = parse(raw)
        if dets: return dets[:4], rgb
        if attempt < max_retries: print('  Retrying...')
    print('No valid detections after retries.')
    return [], rgb

print('Functions loaded.')

In [ ]:
def show_detections(rgb, dets, title=''):
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(rgb); axes[0].set_title(f'Resized ({IMG_SIZE}x{IMG_SIZE})'); axes[0].axis('off')
    axes[1].imshow(rgb); axes[1].set_title(f'{len(dets)} object(s) detected'); axes[1].axis('off')
    for i, d in enumerate(dets):
        b = d['bbox']; c = COLORS[i % len(COLORS)]
        axes[1].add_patch(patches.Rectangle((b[0],b[1]), b[2]-b[0], b[3]-b[1], lw=2.5, ec=c, fc='none'))
        axes[1].text(b[0], b[1]-5, d['label'], fontsize=10, fontweight='bold', color='white',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=c, alpha=0.85))
    if title: fig.suptitle(title, fontsize=15, y=1.02)
    plt.tight_layout(); plt.show()
    print(f'\n{"Object":<25} {"Bounding Box (px)"}')
    print('-' * 50)
    for d in dets: print(f'{d["label"]:<25} {[int(v) for v in d["bbox"]]}')

---
## Test 1: Dog & Cat

In [ ]:
dets, img = detect_objects('sample_dog_cat_house.jpg')
show_detections(img, dets, 'Dog & Cat')

## Test 2: Dog

In [ ]:
dets, img = detect_objects('sample_dog.jpg')
show_detections(img, dets, 'Dog')

## Test 3: Room

In [ ]:
dets, img = detect_objects('sample_room.jpg')
show_detections(img, dets, 'Room Scene')

## Test 4: Landscape

In [ ]:
dets, img = detect_objects('sample_image.jpg')
show_detections(img, dets, 'Landscape')

## Test 5: Your Own Image

In [ ]:
my_image = 'sample_dog.jpg'  # <-- change this
dets, img = detect_objects(my_image)
show_detections(img, dets, 'Custom')

## Test 6: Free-form Q&A

In [ ]:
rgb, b64 = resize_and_encode('sample_dog.jpg')
plt.figure(figsize=(5,5)); plt.imshow(rgb); plt.axis('off'); plt.show()
r = client.complete(messages=[UserMessage(content=[
    TextContentItem(text='What breed is this dog? Describe its posture and surroundings.'),
    ImageContentItem(image_url=ImageUrl(url=f'data:image/jpeg;base64,{b64}')),
])], max_tokens=256, temperature=0.7)
print(r.choices[0].message.content)

## Test 7: Video Frames

In [ ]:
def detect_video_frames(video_path, num_frames=4):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = [int(i * total / num_frames) for i in range(num_frames)]
    fig, axes = plt.subplots(2, num_frames, figsize=(6*num_frames, 12))
    if num_frames == 1: axes = axes.reshape(2,1)
    for col, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        tmp = f'_tmp_{col}.jpg'; cv2.imwrite(tmp, resized)
        axes[0][col].imshow(rgb); axes[0][col].set_title(f'Frame {idx}'); axes[0][col].axis('off')
        print(f'Frame {col+1}/{num_frames}...')
        dets, _ = detect_objects(tmp)
        axes[1][col].imshow(rgb); axes[1][col].set_title(f'{len(dets)} objects'); axes[1][col].axis('off')
        for i, d in enumerate(dets):
            b=d['bbox']; c=COLORS[i%len(COLORS)]
            axes[1][col].add_patch(patches.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],lw=2,ec=c,fc='none'))
            axes[1][col].text(b[0],b[1]-3,d['label'],fontsize=8,fontweight='bold',color='white',
                             bbox=dict(boxstyle='round,pad=0.2',facecolor=c,alpha=0.85))
        os.remove(tmp)
    cap.release(); plt.tight_layout(); plt.show()

# detect_video_frames('your_video.mp4', num_frames=4)